# Module 04, Part 4: The GRU and the limits of recurrence

This notebook is the runnable companion to Part 4 of the Module 04 learning
content. It checks, with code:

1. The gated recurrent unit written by hand matches `nn.GRU`, and has fewer
   parameters than the LSTM.
2. The GRU has the same initialization caveat as the LSTM, on the same task.
3. Recurrence is sequential: time grows with sequence length, and a dependent
   loop is slow compared with one batched operation of the same size.
4. A preview of Module 05: a single direct read over all positions solves the
   delayed-recall task at a length where a gated recurrent network fails.

Recurrent networks are the problem transformers solved. This notebook shows,
with measurements, what that problem looks like. Everything runs on CPU in a few
minutes.

In [1]:
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

INK, BODY, MUTED = "#14161B", "#4E5563", "#7C838C"
NAVY, GREEN, AMBER, KEY, RULE = "#1C3D6E", "#0E6B57", "#8A6A18", "#7A2E8A", "#D9DCE2"

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": RULE, "axes.labelcolor": BODY, "xtick.color": BODY, "ytick.color": BODY,
    "axes.grid": True, "grid.color": RULE, "font.size": 10,
})

torch.manual_seed(0)
print("PyTorch version:", torch.__version__)
count = lambda module: sum(p.numel() for p in module.parameters())

PyTorch version: 2.10.0+cpu


## 1. The GRU, written by hand

The GRU drops the separate cell state and uses two gates, an update gate $z_t$
and a reset gate $r_t$, with a single hidden state:

$$h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$$

PyTorch stores the three weight blocks in the order **reset, update, new**. When
$z_t$ is close to 1, the old state is copied forward almost unchanged.

In [2]:
F, H, T, B = 8, 32, 20, 4
gru = nn.GRU(F, H, batch_first=True)
print("Parameters:  plain RNN", count(nn.RNN(F, H)), "  GRU", count(gru), "  LSTM", count(nn.LSTM(F, H)))

x = torch.randn(B, T, F)
h = torch.zeros(B, H)
outputs = []
for t in range(T):
    gi = x[:, t] @ gru.weight_ih_l0.T + gru.bias_ih_l0
    gh = h @ gru.weight_hh_l0.T + gru.bias_hh_l0
    i_r, i_z, i_n = gi.chunk(3, dim=1)                      # reset, update, new
    h_r, h_z, h_n = gh.chunk(3, dim=1)
    r = torch.sigmoid(i_r + h_r)
    z = torch.sigmoid(i_z + h_z)
    n = torch.tanh(i_n + r * h_n)                           # the reset gate scales the old state's contribution
    h = (1 - z) * n + z * h                                 # a learned blend of new and old
    outputs.append(h)
manual = torch.stack(outputs, dim=1)

out, _ = gru(x)
print("Hand-written GRU matches nn.GRU:", torch.allclose(manual, out, atol=1e-6))

Parameters:  plain RNN 1344   GRU 4032   LSTM 5376
Hand-written GRU matches nn.GRU: True


Three weight blocks instead of four: 4,032 parameters against the LSTM's 5,376,
for the same input and hidden sizes.

## 2. The same initialization caveat

We reuse the delayed-recall task from Part 3: a label at step 0, noise after it,
the label reported at the last step, chance 25%. Same data, optimizer, clipping
and 600 updates for both runs; only the starting update-gate bias differs.

In [3]:
def make_recall_batch(batch_size, seq_len, generator, n_classes=4, noise_dim=2):
    """Delayed recall. Step 0 carries a class label as a one-hot vector in the
    first n_classes features. Every later step carries only noise. The label
    is read out at the LAST step, so the network must carry it across
    seq_len - 1 steps of distraction."""
    labels = torch.randint(0, n_classes, (batch_size,), generator=generator)
    x = torch.randn(batch_size, seq_len, n_classes + noise_dim, generator=generator) * 0.5
    x[:, :, :n_classes] = 0.0
    x[torch.arange(batch_size), 0, labels] = 1.0
    return x, labels


class RecallNet(nn.Module):
    """A recurrent layer followed by a linear classifier that reads the LAST step."""

    def __init__(self, kind, hidden=32, gate_bias=None, n_inputs=6, n_classes=4):
        super().__init__()
        cell = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = cell(n_inputs, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_classes)
        # gate_bias opens the LSTM forget gate or the GRU update gate at the start.
        # PyTorch stores the gates in one block: LSTM (input, forget, cell, output),
        # GRU (reset, update, new). Slice [H:2H] is the forget gate or the update gate.
        # There are two bias vectors; their sum is the effective bias, so split it.
        if gate_bias is not None and kind in ("lstm", "gru"):
            with torch.no_grad():
                self.rnn.bias_ih_l0[hidden:2 * hidden].fill_(gate_bias / 2)
                self.rnn.bias_hh_l0[hidden:2 * hidden].fill_(gate_bias / 2)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])


def train_recall(kind, seq_len, seed=0, gate_bias=None, steps=600, batch_size=64,
                 lr=3e-3, clip=1.0, eval_every=50):
    """Train on the delayed-recall task. Returns (final_accuracy, history)."""
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 100)
    model = RecallNet(kind, gate_bias=gate_bias)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    eval_gen = torch.Generator().manual_seed(999)
    x_val, y_val = make_recall_batch(1000, seq_len, eval_gen)
    history = []
    for step in range(1, steps + 1):
        x, y = make_recall_batch(batch_size, seq_len, gen)
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        if clip:
            nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        if step % eval_every == 0:
            with torch.no_grad():
                history.append((step, (model(x_val).argmax(1) == y_val).float().mean().item()))
    return history[-1][1], history

In [ ]:
runs = {"GRU, default init": None, "GRU, update-gate bias = 3": 3.0}
final = {}
fig, ax = plt.subplots(figsize=(7, 3.4))
for (name, bias), color in zip(runs.items(), (MUTED, NAVY)):
    acc, history = train_recall("gru", seq_len=30, seed=0, gate_bias=bias)
    final[name] = acc
    steps, accs = zip(*history)
    ax.plot(steps, accs, color=color, lw=2, label=name)
ax.axhline(0.25, color=MUTED, ls="--", lw=1)
ax.set_xlabel("training update")
ax.set_ylabel("validation accuracy")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc="center right")
plt.tight_layout()
plt.show()
for name, acc in final.items():
    print(f"{name:28s} accuracy after 600 updates: {acc:.3f}")

The GRU behaves like the LSTM: at default settings it stays at chance within the
budget, and starting with the update gate biased toward keeping the old state
fixes it. Gating helps only when the network starts out using the gate.

## 3. Recurrence is sequential

Step $t$ needs $h_{t-1}$, so the steps cannot run at the same time. We measure two
things. First, how the time for a forward and backward pass grows with sequence
length for an LSTM. Second, the cost of a chain of **dependent** operations
against the same number of **independent** operations done as one batched call.

In [ ]:
def best_of(fn, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)

lstm = nn.LSTM(64, 128, batch_first=True)
lengths = [50, 100, 200, 400]
lstm_times = []
for L in lengths:
    xs = torch.randn(32, L, 64)
    def step():
        o, _ = lstm(xs)
        o.sum().backward()
    lstm_times.append(best_of(step, repeats=3))

print("LSTM forward + backward, batch 32, hidden 128")
for L, s in zip(lengths, lstm_times):
    print(f"  {L:4d} steps: {s * 1000:7.1f} ms   ({s / L * 1000:.3f} ms per step)")

In [ ]:
d, batch = 128, 32
# An orthogonal matrix keeps the values from decaying toward zero; extremely small
# floating-point numbers slow the CPU down and would add noise to the timing.
W = torch.linalg.qr(torch.randn(d, d))[0]
dependent_ms, batched_ms = [], []
for L in lengths:
    x0 = torch.randn(batch, d)
    def dependent_chain():
        h = x0
        for _ in range(L):
            h = torch.tanh(h @ W)          # each step needs the previous one
    X = torch.randn(batch * L, d)
    def one_batched_op():
        torch.tanh(X @ W)                  # the same amount of arithmetic, all at once
    dependent_ms.append(best_of(dependent_chain, 10) * 1000)
    batched_ms.append(best_of(one_batched_op, 10) * 1000)

print(f"{'steps':>6s}{'dependent chain (ms)':>24s}{'one batched op (ms)':>24s}{'ratio':>8s}")
for L, a, b2 in zip(lengths, dependent_ms, batched_ms):
    print(f"{L:6d}{a:24.2f}{b2:24.2f}{a / b2:8.1f}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(lengths, dependent_ms, color=AMBER, lw=2, marker="o", label="dependent chain (recurrence)")
ax.plot(lengths, batched_ms, color=GREEN, lw=2, marker="o", label="one batched operation")
ax.set_xlabel("sequence length")
ax.set_ylabel("time (ms)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The exact milliseconds depend on your machine, and small timings are noisy, so
read the shape rather than the digits: the chain of dependent steps grows in
proportion to its length (double the steps, roughly double the time), and it is
slower than the same arithmetic done at once. On a GPU the gap is
larger, because a GPU is fast only when it has many independent operations to run
together. Recurrence gives it almost none along the time axis.

## 4. A preview of Module 05: one direct read

The second limit is that everything must travel through one fixed-size state, and
distance equals path length. A gated network with open gates still has to carry
the label through every intermediate step. What if the final prediction could look
**directly** at every position instead?

Below is a minimal sketch: each position gets a score, a softmax turns the scores
into weights that sum to 1, and the prediction reads the weighted sum of the
positions. There is no recurrence at all. This is not the full attention mechanism;
Module 05 builds that. It is enough to show the effect.

In [ ]:
class DirectRead(nn.Module):
    def __init__(self, n_inputs=6, hidden=32, n_classes=4):
        super().__init__()
        self.key = nn.Linear(n_inputs, hidden)
        self.value = nn.Linear(n_inputs, hidden)
        self.query = nn.Parameter(torch.randn(hidden) * 0.1)
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        scores = torch.tanh(self.key(x)) @ self.query            # one score per position
        weights = torch.softmax(scores, dim=1)                   # (batch, time), sums to 1
        return self.head((weights.unsqueeze(-1) * self.value(x)).sum(dim=1))


def train_direct_read(seq_len, seed=0, steps=600):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 100)
    model = DirectRead()
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = make_recall_batch(64, seq_len, gen)
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    x_val, y_val = make_recall_batch(1000, seq_len, torch.Generator().manual_seed(999))
    with torch.no_grad():
        return (model(x_val).argmax(1) == y_val).float().mean().item()


seq_len = 200
acc_direct = train_direct_read(seq_len)
acc_lstm, _ = train_recall("lstm", seq_len=seq_len, seed=0, gate_bias=3.0)
print(f"Delayed recall over {seq_len} steps, 600 updates each (chance is 0.25):")
print(f"  LSTM with forget bias 3 (recurrent, open gate): {acc_lstm:.3f}")
print(f"  Direct read over all positions (no recurrence):  {acc_direct:.3f}")

At 30 and 60 steps, the open-gate LSTM and GRU solved this task in Part 3 and in
section 2 above. At 200 steps, within the same 600 updates, the open-gate LSTM stays
at chance while a single direct read over all positions solves it. That is the
problem transformers solved: any two positions are one hop apart, and no position
has to wait for the one before it.

## Try it

1. Rerun section 4 with `seq_len = 100`. Does the open-gate LSTM learn? Try seeds
   0 and 1. What does that say about reliability at moderate lengths?
2. Give the LSTM 3,000 updates instead of 600 at length 200. Does more training
   close the gap, or does the direct read still win on effort?
3. In section 3, double the hidden size from 128 to 256 and rerun the LSTM timing.
   Does the time per step grow in proportion?
4. Write down, in your own words, the two separate things (speed and fixed-size
   memory) that prevent recurrence from scaling to very long sequences, and which
   of them gating can help with.

## What this notebook showed

- The hand-written GRU, with PyTorch's gate order (reset, update, new), matches
  `nn.GRU`, with 4,032 parameters against the LSTM's 5,376.
- The GRU needs an open update gate in the same way the LSTM needs an open forget
  gate: at default settings it stays at chance on delayed recall over 30 steps.
- Recurrence is sequential: LSTM time grows with sequence length, and a chain of
  dependent steps is slower than the same arithmetic done as one batched call.
- With no recurrence at all, one direct read over every position solves a
  200-step delayed recall that an open-gate LSTM does not learn in the same
  number of updates. Module 05 develops this idea into attention.